In [1]:
!source /ws_slam/setup_env.sh
!ros2 topic list

/battery_state
/cmd_vel
/events/write_split
/imu
/joint_states
/magnetic_field
/odom
/parameter_events
/robot_description
/rosout
/scan
/sensor_state
/tf
/tf_static


In [2]:
import time
import pandas as pd
import rclpy

from rclpy.node import Node
from geometry_msgs.msg import Twist


class VelCmdPlayer(Node):
    def __init__(self, csv_file):
        super().__init__('vel_cmd_player')

        self.pub = self.create_publisher(Twist, '/cmd_vel', 10)

        df = pd.read_csv(csv_file)

        self.timestamps = df['time'].to_numpy()
        self.v_cmds = df['v_cmd'].to_numpy()
        self.w_cmds = df['w_cmd'].to_numpy()

        self.idx = 0
        self.start_time = time.monotonic()

        # 100 Hz
        self.timer = self.create_timer(0.05, self.timer_callback)

    def timer_callback(self):
        elapsed = time.monotonic() - self.start_time

        # # Advance index without publishing
        # if not (
        #     self.idx + 1 < len(self.timestamps)
        #     and elapsed >= self.timestamps[self.idx + 1]
        # ):
        #     return

        msg = Twist()

        if self.idx < len(self.timestamps):
            msg.linear.x = float(self.v_cmds[self.idx])
            msg.angular.z = float(self.w_cmds[self.idx])

        self.pub.publish(msg)

        self.idx += 1

        print(f"Published cmd at time {elapsed:.2f}s: v={msg.linear.x:.2f}, w={msg.angular.z:.2f}")

        # stop after final timestamp
        if elapsed > self.timestamps[-1]:
            self.publish_zero()
            rclpy.shutdown()

    def publish_zero(self):
        msg = Twist()
        self.pub.publish(msg)

pd.read_csv("vel_cmd_filt.csv")

,time,v_cmd,w_cmd
0,0.00,0.000000e+00,0.000000
1,0.05,0.000000e+00,0.000000
2,0.10,0.000000e+00,0.000000
3,0.15,0.000000e+00,0.000000
4,0.20,0.000000e+00,0.000000
...,...,...,...
1196,59.80,8.613610e-07,-0.000007
1197,59.85,7.830555e-07,-0.000007
1198,59.90,7.118686e-07,-0.000006
1199,59.95,6.471533e-07,-0.000005


In [ ]:

rclpy.init()

node = VelCmdPlayer("vel_cmd_filt.csv")

try:
    rclpy.spin(node)
except KeyboardInterrupt:
    node.publish_zero()
finally:
    if rclpy.ok():
        node.publish_zero()
        rclpy.shutdown()
    node.destroy_node()

Published cmd at time 0.05s: v=0.00, w=0.00
Published cmd at time 0.10s: v=0.00, w=0.00
Published cmd at time 0.15s: v=0.00, w=0.00
Published cmd at time 0.20s: v=0.00, w=0.00
Published cmd at time 0.25s: v=0.00, w=0.00
Published cmd at time 0.30s: v=0.00, w=0.00
Published cmd at time 0.35s: v=0.00, w=0.00
Published cmd at time 0.40s: v=0.00, w=0.00
Published cmd at time 0.45s: v=0.00, w=0.00
Published cmd at time 0.50s: v=0.00, w=0.00
Published cmd at time 0.55s: v=0.00, w=0.00
Published cmd at time 0.60s: v=0.00, w=0.00
Published cmd at time 0.65s: v=0.00, w=0.00
Published cmd at time 0.70s: v=0.00, w=0.00
Published cmd at time 0.75s: v=0.00, w=0.00
Published cmd at time 0.80s: v=0.00, w=0.00
Published cmd at time 0.85s: v=0.00, w=0.00
Published cmd at time 0.90s: v=0.00, w=0.00
Published cmd at time 0.95s: v=0.00, w=0.00
Published cmd at time 1.00s: v=0.00, w=0.00
Published cmd at time 1.05s: v=0.00, w=0.00
Published cmd at time 1.10s: v=0.00, w=0.00
Published cmd at time 1.15s: v=0

1781187199.310489 [66]        tev: ddsi_udp_conn_write to udp/192.168.123.15:42288 failed with retcode -1
1781187199.409719 [66]        tev: ddsi_udp_conn_write to udp/239.255.0.1:23900 failed with retcode -1
1781187199.409744 [66]        tev: ddsi_udp_conn_write to udp/192.168.123.15:42288 failed with retcode -1
1781187199.409755 [66]        tev: ddsi_udp_conn_write to udp/192.168.123.15:48360 failed with retcode -1
1781187199.409765 [66]        tev: ddsi_udp_conn_write to udp/192.168.123.59:23910 failed with retcode -1
1781187199.409773 [66]        tev: ddsi_udp_conn_write to udp/192.168.123.59:23912 failed with retcode -1
1781187199.409782 [66]        tev: ddsi_udp_conn_write to udp/192.168.123.59:23914 failed with retcode -1
1781187199.409792 [66]        tev: ddsi_udp_conn_write to udp/192.168.123.59:23916 failed with retcode -1
1781187199.410491 [66]        tev: ddsi_udp_conn_write to udp/192.168.123.15:42288 failed with retcode -1
1781187199.510599 [66]        tev: ddsi_udp_conn_